# OLYMPIC AI 2026 - TÁC VỤ 1: TRÍCH XUẤT BỘ BA KHÍA CẠNH - DANH MỤC - CẢM XÚC (ACSTE)
## Notebook: Quy trình Sinh file kết quả dự đoán (generate_result.ipynb)

Notebook này thực thi toàn bộ luồng suy luận (inference) từ dữ liệu kiểm tra (`PublicTest.json` / `PrivateTest.json`), sử dụng mô hình ngôn ngữ lớn **Qwen3-0.6B (LoRA Fine-tuned)** để sinh chuỗi bộ ba `aspect | category | sentiment`, sau đó bóc tách và đóng gói thành file `submission.json` đúng chuẩn của Ban Tổ Chức.

* **Mô hình**: `Qwen3-0.6B` (Causal LM, 28 layers, LoRA SFT)
* **5 Domains**: Restaurant, Laptop, Hotel, Books, Clothing
* **Target Output**: `submission.json` gồm 5 domain keys với danh sách các triples
* **Metric mục tiêu**: Micro-F1 (Exact Triple Match)


In [ ]:
# 1. Khai báo các thư viện cần thiết
import os
import re
import json
import torch
from pathlib import Path
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"PyTorch version: {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Thiết bị thực thi: {device.upper()}")


In [ ]:
# 2. Cấu hình đường dẫn dữ liệu và mô hình
ALL_DOMAINS = ["Restaurant", "Laptop", "Hotel", "Books", "Clothing"]
FIELD_SEP = " | "

def resolve_paths():
    candidates_data = [
        Path("data/TACVU1"),
        Path("../data/TACVU1"),
        Path("./data"),
        Path(".")
    ]
    data_root = None
    for c in candidates_data:
        if (c / "public_test/PublicTest.json").exists() or (c / "train/Train.json").exists():
            data_root = c
            break
    if data_root is None:
        data_root = Path("data/TACVU1")

    candidates_model = [
        Path("outputs/qwen_all/checkpoint-best"),
        Path("../outputs/qwen_all/checkpoint-best"),
        Path("checkpoint-best"),
        Path("models/qwen3-0.6B")
    ]
    model_dir = None
    for m in candidates_model:
        if m.exists():
            model_dir = m
            break
    if model_dir is None:
        model_dir = Path("outputs/qwen_all/checkpoint-best")

    return data_root, model_dir

data_root, model_dir = resolve_paths()
print(f"Data root: {data_root.resolve()}")
print(f"Model path: {model_dir.resolve()}")


In [ ]:
# 3. Định nghĩa Prompt Template & Parser cho ACSTE
def build_prompt(raw_words: str, domain: str) -> str:
    system_msg = (
        "You are an expert in Aspect-Based Sentiment Analysis (ABSA). "
        "Extract all aspect-category-sentiment triples from the review sentence in the specified domain. "
        "Output format: aspect | category | sentiment. "
        "Separate multiple triples with ' ; '. If no triple, output <empty>. "
        "Output ONLY the triples without explanation or thinking tokens."
    )
    user_msg = f"Domain: {domain}\nReview: {raw_words}"
    return (
        f"<|im_start|>system\n{system_msg}<|im_end|>\n"
        f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

def parse_generated_triples(text: str) -> list:
    if not text:
        return []
    t = text.strip()
    if not t or t.lower() in ("<empty>", "none", "n/a", "empty"):
        return []
    triples = []
    for chunk in re.split(r"\s*;\s*", t):
        chunk = chunk.strip()
        if not chunk:
            continue
        parts = chunk.split(FIELD_SEP, 2)
        if len(parts) != 3:
            continue
        aspect, category, sentiment = (p.strip() for p in parts)
        aspect_clean = "NULL" if aspect.lower() == "null" or not aspect else aspect
        sentiment_clean = sentiment.upper() if sentiment.upper() in ("POS", "NEG", "NEU") else "POS"
        triples.append({
            "aspect": aspect_clean,
            "category": category,
            "sentiment": sentiment_clean
        })
    return triples


In [ ]:
# 4. Nạp Model & Tokenizer
print(f"Đang nạp mô hình từ {model_dir}...")
tokenizer = AutoTokenizer.from_pretrained(str(model_dir), local_files_only=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model_dtype = torch.float32 if device == "cpu" else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    str(model_dir),
    local_files_only=True,
    dtype=model_dtype
).to(device)
model.eval()
print("Mô hình đã sẵn sàng!")


In [ ]:
# 5. Đọc tập dữ liệu kiểm tra (Ưu tiên PrivateTest nếu có, ngược lại PublicTest)
import zipfile

# Tự động giải nén private_test.zip nếu có
priv_zip = data_root / "private_test.zip"
if priv_zip.exists() and not (data_root / "private_test/PrivateTest.json").exists():
    try:
        with zipfile.ZipFile(priv_zip) as z:
            z.extractall(data_root, pwd=b"ckolp26@vku")
    except Exception as e:
        pass

priv_candidates = [
    data_root / "private_test/PrivateTest.json",
    data_root / "PrivateTest.json",
    Path("/app/input/res/PrivateTest.json"),
    Path("PrivateTest.json")
]
test_file = None
for p in priv_candidates:
    if p.exists():
        test_file = p
        test_split = "PrivateTest"
        break

if test_file is None:
    pub_candidates = [
        data_root / "public_test/PublicTest.json",
        data_root / "PublicTest.json",
        Path("PublicTest.json")
    ]
    for p in pub_candidates:
        if p.exists():
            test_file = p
            test_split = "PublicTest"
            break

print(f"Đang đọc dữ liệu [{test_split}] từ: {test_file}")
with open(test_file, encoding="utf-8") as f:
    test_data = json.load(f)

for d in ALL_DOMAINS:
    print(f"Domain {d}: {len(test_data[d])} câu")


In [ ]:
# 6. Thực thi suy luận trên 5 domains
submission = {}

for domain in ALL_DOMAINS:
    items = test_data[domain]
    pred_list = []
    print(f"\nĐang sinh kết quả cho domain [{domain}] ({len(items)} câu)...")
    
    for item in tqdm(items):
        raw_words = item["raw_words"]
        prompt = build_prompt(raw_words, domain)
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        input_len = inputs.input_ids.shape[1]
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        
        generated_tokens = outputs[0][input_len:]
        response_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
        triples = parse_generated_triples(response_text)
        
        pred_list.append({
            "raw_words": raw_words,
            "triples": triples
        })
    
    submission[domain] = pred_list

print("\nHoàn tất toàn bộ suy luận!")


In [ ]:
# 7. Xuất file submission.json và kiểm tra tính hợp lệ
output_file = Path("submission.json")
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(submission, f, indent=2, ensure_ascii=False)

print(f"Đã lưu file nộp bài tại: {output_file.resolve()}")

# Kiểm tra tính toàn vẹn
assert list(submission.keys()) == ALL_DOMAINS, "Thiếu key domain!"
for d in ALL_DOMAINS:
    assert len(submission[d]) == len(test_data[d]), f"Lệch số lượng câu ở domain {d}!"
print("Tất cả kiểm tra đều HỢP LỆ và sẵn sàng nộp bài!")
